In [1]:
# --- GPU + Repository Setup ---
from pathlib import Path

REPO_URL = "https://github.com/tomasonjo/kg-rag"
REPO_DIR = Path("/content/kg-rag")

print("🔍 Checking GPU...")
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\n📂 Setting up repository...")

🔍 Checking GPU...
CUDA available: True
GPU: Tesla T4

📂 Setting up repository...


In [24]:
%cd /content

/content


In [25]:
%pwd

'/content'

In [2]:
# Clone repo if not exists
if not REPO_DIR.exists():
    !git clone {REPO_URL}

%cd {REPO_DIR}
!git pull

Cloning into 'kg-rag'...
remote: Enumerating objects: 162, done.
remote: Counting objects: 100% (162/162), done.
remote: Compressing objects: 100% (118/118), done.
remote: Total 162 (delta 90), reused 101 (delta 38), pack-reused 0 (from 0)
Receiving objects: 100% (162/162), 114.53 KiB | 6.74 MiB/s, done.
Resolving deltas: 100% (90/90), done.
/content/kg-rag
Already up to date.


In [3]:

print("\n📦 Installing dependencies...")
!pip install -q -r notebooks/requirements.txt || pip install -q neo4j sentence-transformers transformers accelerate tiktoken python-dotenv

print("\n✅ Setup complete.")



📦 Installing dependencies...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.7/67.7 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.1/225.1 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 113.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.4/325.4 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 124.3 MB/s eta 0:00:00

✅ Setup complete.


In [9]:
!pip install -q "transformers>=4.42" accelerate sentencepiece sentence-transformers neo4j bitsandbytes


In [4]:
# --- Neo4j Aura Credentials ---
import os
NEO4J_URI      = os.getenv("NEO4J_URI",      "neo4j+s://5db7d1f6.databases.neo4j.io")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "hCBE_-lRkn_A_02LIVIpnppdQ9V6n0BXVW6rYuC7srA")

# NEO4J_URI = "neo4j+s://5db7d1f6.databases.neo4j.io"
# NEO4J_USERNAME= "neo4j"
# NEO4J_PASSWORD = "hCBE_-lRkn_A_02LIVIpnppdQ9V6n0BXVW6rYuC7srA"   # <--- CHANGE THIS

print("🔐 Neo4j environment variables set.")


🔐 Neo4j environment variables set.


In [29]:
# # --- Imports from your codebase ---
from notebooks.utils import neo4j_driver, embed, chat, chunk_text

# # Quick test
# with neo4j_driver.session() as session:
#     print(session.run("RETURN 1 AS ok").single())

# print("\n🔮 Embedding test:", len(embed(["hello world"])[0]))


ValueError: ❌ NEO4J_URI is missing. Check your .env or Codespace variables.

In [5]:
import requests

remote_pdf_url = "https://arxiv.org/pdf/1709.00666.pdf"
pdf_filename = "ch02-downloaded.pdf"

response = requests.get(remote_pdf_url)

if response.status_code == 200:
    with open(pdf_filename, "wb") as pdf_file:
        pdf_file.write(response.content)
else:
    print("Failed to download the PDF. Status code:", response.status_code)



In [ ]:
%pwd

'/content/kg-rag'

In [6]:

import pdfplumber

text = ""

with pdfplumber.open(pdf_filename) as pdf:
    for page in pdf.pages:
        text += page.extract_text()

print(text[0:20])

Einstein’s Patents a


In [7]:
# from utils import chunk_text

chunks = chunk_text(text, 500, 40)
print(len(chunks))
print(chunks[0])

NameError: name 'chunk_text' is not defined

In [ ]:
from sentence_transformers import SentenceTransformer

# Load a local embedding model (no API needed)
local_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def embed(texts):
    """
    Work exactly like your OpenAI embed() function,
    but using a free, local embedding model.
    """
    return local_model.encode(texts).tolist()


In [ ]:
embeddings = embed(chunks)

print(embeddings[0][0:3])          # preview numbers
print(len(embeddings))             # number of chunks
print(len(embeddings[0]))          # should print 384

[-0.04776293784379959, 0.06253520399332047, -0.04059496894478798]
89
384


In [ ]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD),
    notifications_min_severity="OFF"
)


In [ ]:
driver.execute_query("""CREATE VECTOR INDEX pdf IF NOT EXISTS
FOR (c:Chunk)
ON c.embedding""")

EagerResult(records=[], summary=<neo4j._work.summary.ResultSummary object at 0x7b7900583a40>, keys=[])

In [ ]:
# Add to neo4j
cypher_query = '''
WITH $chunks as chunks, range(0, size($chunks)) AS index
UNWIND index AS i
WITH i, chunks[i] AS chunk, $embeddings[i] AS embedding
MERGE (c:Chunk {index: i})
SET c.text = chunk, c.embedding = embedding
'''

driver.execute_query(cypher_query, chunks=chunks, embeddings=embeddings)

EagerResult(records=[], summary=<neo4j._work.summary.ResultSummary object at 0x7b7902acca10>, keys=[])

In [ ]:
records, _, _ = driver.execute_query("MATCH (c:Chunk) WHERE c.index = 0 RETURN c.embedding, c.text")

print(records[0]["c.text"][0:30])
print(records[0]["c.embedding"][0:3])

Einstein’s Patents and Inventi
[-0.04776293784379959, 0.06253520399332047, -0.04059496894478798]


In [ ]:
question = "At what time was Einstein really interested in experimental works?"
question_embedding = embed([question])[0]

query = '''
CALL db.index.vector.queryNodes('pdf', $k, $question_embedding) YIELD node AS hits, score
RETURN hits.text AS text, score, hits.index AS index
'''
similar_records, _, _ = driver.execute_query(query, question_embedding=question_embedding, k=4)

for record in similar_records:
    print(record["text"])
    print(record["score"], record["index"])
    print("======")

Albert Einstein topped the list.
Einstein’s choice as the person of the century didn’t invoke any resentment, it was generally agreed
that 20th century is the age of Science and undoubtedly, Einstein’s contribution to Science, to the
understanding of the intricate laws of nature was unparalleled. He greatly influenced modern science;
altered our views on space‐time, matter and energy, gave new interpretation to gravity etc. The
enormous popularity he enjoyed during his lifetime and even now, is rare for any individual; religious
leader, politician,
0.8008778095245361 2
Einstein’s life was rather featureless. He diligently worked at the patent office,
played violin, discussed physics with his friends, write few not so interesting papers. Then in 1905, he
took the academic world by surprise. In the annals of physics, the year 1905 is known as “annus
mirabilis” or the year of miracle. Indeed, a miracle happened. Albert Einstein, barely 26 years old,
sitting in an obscure Swiss patent offi

In [ ]:
# # =========================
# # Hybrid RAG + Neo4j + Phi-3 (Vector + Keyword)
# # =========================
# # pip install -q transformers accelerate sentencepiece sentence-transformers neo4j

# import logging, time
# from threading import Thread

# import numpy as np
# import torch
# from neo4j import GraphDatabase
# from sentence_transformers import SentenceTransformer
# from transformers import AutoModelForCausalLM, AutoTokenizer, TextIteratorStreamer

# # ---------- Logging ----------
# logging.basicConfig(
#     level=logging.INFO,
#     format="%(asctime)s [%(levelname)s] [%(name)s] %(message)s",
# )
# log = logging.getLogger("HYBRID-RAG")

# STEP_TIMES = {}
# MAX_CONTEXT_TOKENS = 4000

# def record_step(name, start):
#     STEP_TIMES[name] = time.perf_counter() - start

# def print_summary():
#     log.info("\n========== PIPELINE TIMING SUMMARY ==========")
#     total = 0
#     for step, sec in STEP_TIMES.items():
#         log.info(f"{step:28s}: {sec:.3f}s")
#         total += sec
#     log.info("---------------------------------------------")
#     log.info(f"TOTAL PIPELINE TIME         : {total:.3f}s")
#     log.info("=============================================\n")


# # =====================================================
# # STEP 1: Environment + Neo4j + Embeddings
# # =====================================================
# step = time.perf_counter()

# device = "cuda" if torch.cuda.is_available() else "cpu"
# log.info(f"Device selected: {device}")
# if device == "cuda":
#     log.info(f"GPU: {torch.cuda.get_device_name(0)}")
#     log.info(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.2f} GB")
# else:
#     log.warning("Running on CPU — LLM inference will be slow.")

# import os
# NEO4J_URI      = os.getenv("NEO4J_URI",      "neo4j+s://YOUR_AURA.databases.neo4j.io")
# NEO4J_USERNAME = os.getenv("NEO4J_USERNAME", "neo4j")
# NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "password")

# neo4j_driver = GraphDatabase.driver(
#     NEO4J_URI,
#     auth=(NEO4J_USERNAME, NEO4J_PASSWORD),
# )

# EMB_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
# emb_model = SentenceTransformer(EMB_MODEL_NAME)
# EMB_DIM = emb_model.get_sentence_embedding_dimension()
# log.info(f"Embedding model: {EMB_MODEL_NAME} (dim={EMB_DIM})")

# record_step("Env + Neo4j + Embeddings", step)


# def embed(texts):
#     """Return list of embeddings (list[list[float]])."""
#     t0 = time.perf_counter()
#     vecs = emb_model.encode(texts, normalize_embeddings=False)
#     t1 = time.perf_counter()
#     log.info(f"Embedded {len(texts)} text(s) in {t1 - t0:.3f}s")
#     return vecs.tolist()

# def cosine_similarity(u, v):
#     u = np.array(u, dtype=np.float32)
#     v = np.array(v, dtype=np.float32)
#     num = float(np.dot(u, v))
#     den = float(np.linalg.norm(u) * np.linalg.norm(v) + 1e-9)
#     return num / den


# # =====================================================
# # STEP 2: LLM (Phi-3) Setup
# # =====================================================
# step = time.perf_counter()

# MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"
# log.info(f"Loading LLM: {MODEL_NAME}")

# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# model = AutoModelForCausalLM.from_pretrained(
#     MODEL_NAME,
#     dtype=torch.float16 if device == "cuda" else torch.float32,
# )
# model.to(device)
# model.eval()

# num_params = sum(p.numel() for p in model.parameters())
# log.info(f"LLM params: {num_params/1e9:.3f}B")

# record_step("LLM Loading", step)


# # =====================================================
# # STEP 3: HYBRID Neo4j Query (Vector + Full-text)
# # =====================================================

# HYBRID_QUERY = """
# CALL {
#     // vector index
#     CALL db.index.vector.queryNodes('pdf', $k, $question_embedding) YIELD node, score
#     WITH collect({node:node, score:score}) AS nodes, max(score) AS max
#     UNWIND nodes AS n
#     RETURN n.node AS node, (n.score / max) AS score

#     UNION

#     // keyword index (full-text)
#     CALL db.index.fulltext.queryNodes('ftPdfChunk', $question, {limit: $k})
#     YIELD node, score
#     WITH collect({node:node, score:score}) AS nodes, max(score) AS max
#     UNWIND nodes AS n
#     RETURN n.node AS node, (n.score / max) AS score
# }
# // dedup & combine
# WITH node, max(score) AS score
# ORDER BY score DESC
# LIMIT $k
# RETURN node, score
# """

# def hybrid_search_neo4j(question: str, k: int = 4):
#     """
#     Hybrid retrieval:
#       - Vector search on embedding index 'pdf'
#       - Fulltext search on 'ftPdfChunk'
#       - Scores normalized and merged
#     Also computes cosine similarity for inspection.
#     """
#     step = time.perf_counter()
#     log.info(f"STEP: Hybrid Search — question={question!r}, k={k}")

#     question_embedding = embed([question])[0]

#     records, summary, keys = neo4j_driver.execute_query(
#         HYBRID_QUERY,
#         question_embedding=question_embedding,
#         question=question,
#         k=k,
#     )

#     hybrid_records = []
#     for i, rec in enumerate(records):
#         node = rec["node"]
#         text = node["text"]
#         hybrid_score = rec["score"]          # normalized hybrid score
#         node_emb = node.get("embedding")     # assumes embedding stored on node

#         cos_sim = None
#         if node_emb is not None:
#             cos_sim = cosine_similarity(question_embedding, node_emb)

#         hybrid_records.append(
#             {
#                 "text": text,
#                 "score": hybrid_score,
#                 "cosine_similarity": cos_sim,
#                 "node": node,
#             }
#         )
#         log.info(
#             f"HybridHit[{i}]: hybrid_score={hybrid_score:.4f}, "
#             f"cos_sim={cos_sim:.4f} len={len(text)}"
#             if cos_sim is not None
#             else f"HybridHit[{i}]: hybrid_score={hybrid_score:.4f}, len={len(text)}"
#         )

#     record_step("Hybrid Search (Neo4j)", step)
#     return hybrid_records


# # =====================================================
# # STEP 4: RAG Prompt Building
# # =====================================================
# def build_rag_prompt(similar_records, question: str):
#     step = time.perf_counter()
#     log.info("STEP: Build Prompt — combining docs + question")

#     docs = [r["text"] for r in similar_records]
#     for i, d in enumerate(docs[:3]):
#         log.info(f"Doc[{i}] length: {len(d)} chars")

#     docs_block = "\n\n---\n\n".join(docs)

#     system_message = (
#         "You are a helpful assistant. You must ONLY use the provided documents. "
#         "If the answer is not in them, say you don't know."
#     )

#     user_message = f"""
# Use the following documents to answer the question that will follow:

# {docs_block}

# ---

# The question to answer using ONLY the above documents is:
# {question}
# """.strip()

#     log.info(f"User message length: {len(user_message)} chars")
#     record_step("Build Prompt", step)
#     return system_message, user_message


# # =====================================================
# # STEP 5: Tokenization + Context Check
# # =====================================================
# def tokenize_and_check(prompt: str):
#     step = time.perf_counter()
#     log.info("STEP: Tokenization + Context Check")

#     inputs = tokenizer(prompt, return_tensors="pt")
#     seq_len = inputs["input_ids"].shape[1]
#     pct = seq_len / MAX_CONTEXT_TOKENS * 100

#     log.info(f"Prompt tokens: {seq_len}/{MAX_CONTEXT_TOKENS} ({pct:.1f}%)")
#     if seq_len > MAX_CONTEXT_TOKENS:
#         log.warning("⚠️ Prompt exceeds context window — will be truncated.")
#     elif pct > 80:
#         log.warning("⚠️ Prompt uses >80% of context — little room to generate.")

#     record_step("Tokenization", step)
#     return inputs.to(device), seq_len


# # =====================================================
# # STEP 6: Streaming LLM Inference
# # =====================================================
# def local_stream(system_message, user_message, max_new_tokens=256):
#     step = time.perf_counter()
#     log.info("STEP: Inference — starting generation")

#     prompt = f"<|system|>\n{system_message}\n<|user|>\n{user_message}\n<|assistant|>\n"
#     inputs, _ = tokenize_and_check(prompt)

#     streamer = TextIteratorStreamer(tokenizer, skip_special_tokens=True)
#     gen_kwargs = dict(
#         **inputs,
#         streamer=streamer,
#         max_new_tokens=max_new_tokens,
#         do_sample=False,
#         temperature=0.1,
#         pad_token_id=tokenizer.eos_token_id,
#     )

#     def _gen():
#         try:
#             model.generate(**gen_kwargs)
#         except Exception as e:
#             log.exception(f"Generation error: {e}")

#     Thread(target=_gen, daemon=True).start()

#     t0 = time.perf_counter()
#     token_count = 0
#     char_count = 0

#     for piece in streamer:
#         print(piece, end="", flush=True)
#         token_count += 1
#         char_count += len(piece)

#     t1 = time.perf_counter()
#     elapsed = t1 - t0
#     tps = token_count / elapsed if elapsed else 0.0

#     log.info(f"\nGenerated tokens: {token_count}, chars: {char_count}")
#     log.info(f"Inference time: {elapsed:.2f}s → {tps:.2f} tokens/s")

#     record_step("Inference", step)


# # =====================================================
# # STEP 7: Full Hybrid RAG Wrapper
# # =====================================================
# def answer_with_hybrid_rag(question: str, k: int = 4):
#     """
#     End-to-end:
#       1) embed question
#       2) hybrid retrieval (vector + fulltext) from Neo4j
#       3) build RAG prompt
#       4) stream answer from Phi-3
#     """
#     log.info("STEP: Pipeline Start — Hybrid RAG Query")
#     similar_hybrid_records = hybrid_search_neo4j(question, k=k)

#     if not similar_hybrid_records:
#         log.warning("No hybrid records returned from Neo4j.")
#         return

#     system_msg, user_msg = build_rag_prompt(similar_hybrid_records, question)

#     print("\n📌 QUESTION:", question)
#     print("\n🧠 ANSWER (Hybrid RAG):\n")
#     local_stream(system_msg, user_msg)


# # =====================================================
# # Example usage
# # =====================================================
# if __name__ == "__main__":
#     q = "Explain the main idea of the document."
#     answer_with_hybrid_rag(q, k=4)
#     print_summary()


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

: 

In [ ]:
# =========================
# Hybrid RAG + Neo4j + HF LLM (Fast/Quality Modes, 4-bit, Full Logging)
# =========================
# Run once in Colab:
# !pip install -q transformers accelerate sentencepiece sentence-transformers neo4j bitsandbytes

import logging, time, os
from threading import Thread

import numpy as np
import torch
from neo4j import GraphDatabase
from sentence_transformers import SentenceTransformer
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TextIteratorStreamer,
)

try:
    from transformers import BitsAndBytesConfig
    BNB_AVAILABLE = True
except ImportError:
    BNB_AVAILABLE = False

# ---------- CONFIG TO TUNE SPEED vs QUALITY ----------
FAST_MODE = True          # True = TinyLlama (very fast), False = Phi-3 (better quality)
QUANTIZE_4BIT = True      # Only used when FAST_MODE = False and GPU + bitsandbytes available
MAX_NEW_TOKENS_FAST = 96
MAX_NEW_TOKENS_QUALITY = 256
MAX_DOC_CHARS = 1000      # truncate each chunk before sending to LLM
MAX_CONTEXT_TOKENS = 4000 # Phi-3 mini context; TinyLlama is similar scale

# ---------- Logging ----------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] [%(name)s] %(message)s",
)
log = logging.getLogger("HYBRID-RAG")

STEP_TIMES = {}

def record_step(name, start):
    STEP_TIMES[name] = time.perf_counter() - start

def print_summary():
    log.info("\n========== PIPELINE TIMING SUMMARY ==========")
    total = 0.0
    for step, sec in STEP_TIMES.items():
        log.info(f"{step:30s}: {sec:.3f}s")
        total += sec
    log.info("----------------------------------------------")
    log.info(f"TOTAL PIPELINE TIME            : {total:.3f}s")
    log.info("==============================================\n")


# =====================================================
# STEP 1: Environment + Neo4j + Embeddings
# =====================================================
step = time.perf_counter()

device = "cuda" if torch.cuda.is_available() else "cpu"
log.info(f"Device selected: {device}")
if device == "cuda":
    log.info(f"GPU: {torch.cuda.get_device_name(0)}")
    log.info(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.2f} GB")
else:
    log.warning("Running on CPU — expect slow LLM inference, especially in QUALITY mode.")

# NEO4J_URI      = os.getenv("NEO4J_URI",      "neo4j+s://YOUR_AURA.databases.neo4j.io")
# NEO4J_USERNAME = os.getenv("NEO4J_USERNAME", "neo4j")
# NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "password")

neo4j_driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD),
)

EMB_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
emb_model = SentenceTransformer(EMB_MODEL_NAME)
EMB_DIM = emb_model.get_sentence_embedding_dimension()
log.info(f"Embedding model: {EMB_MODEL_NAME} (dim={EMB_DIM})")

record_step("Env + Neo4j + Embeddings", step)


def embed(texts):
    """Return list of embeddings (list[list[float]])."""
    t0 = time.perf_counter()
    vecs = emb_model.encode(texts, normalize_embeddings=False)
    t1 = time.perf_counter()
    log.info(f"Embedded {len(texts)} text(s) in {t1 - t0:.3f}s")
    return vecs.tolist()

def cosine_similarity(u, v):
    u = np.array(u, dtype=np.float32)
    v = np.array(v, dtype=np.float32)
    num = float(np.dot(u, v))
    den = float(np.linalg.norm(u) * np.linalg.norm(v) + 1e-9)
    return num / den


# =====================================================
# STEP 2: LLM Setup (FAST/QUALITY + 4-bit)
# =====================================================
step = time.perf_counter()

if FAST_MODE:
    MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
    MAX_NEW_TOKENS = MAX_NEW_TOKENS_FAST
    log.info("LLM mode: FAST")
else:
    MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"
    MAX_NEW_TOKENS = MAX_NEW_TOKENS_QUALITY
    log.info("LLM mode: QUALITY")

log.info(f"Loading LLM: {MODEL_NAME} (FAST_MODE={FAST_MODE}, QUANTIZE_4BIT={QUANTIZE_4BIT})")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if not FAST_MODE and QUANTIZE_4BIT and device == "cuda" and BNB_AVAILABLE:
    log.info("Using 4-bit quantization (bitsandbytes) for QUALITY mode.")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
    )
else:
    if QUANTIZE_4BIT and not BNB_AVAILABLE and not FAST_MODE:
        log.warning("QUANTIZE_4BIT=True but bitsandbytes not available — falling back to normal fp16/fp32.")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        dtype=torch.float16 if device == "cuda" else torch.float32,
    )
    model.to(device)

model.eval()

num_params = sum(p.numel() for p in model.parameters())
log.info(f"LLM params: {num_params/1e9:.3f}B")

record_step("LLM Loading", step)


# =====================================================
# STEP 3: Hybrid Neo4j Search (Vector + Full-text)
# =====================================================
HYBRID_QUERY = """
CALL {
    // vector index
    CALL db.index.vector.queryNodes('pdf', $k, $question_embedding) YIELD node, score
    WITH collect({node:node, score:score}) AS nodes, max(score) AS max
    UNWIND nodes AS n
    RETURN n.node AS node, (n.score / max) AS score

    UNION

    // keyword index (full-text)
    CALL db.index.fulltext.queryNodes('ftPdfChunk', $question, {limit: $k})
    YIELD node, score
    WITH collect({node:node, score:score}) AS nodes, max(score) AS max
    UNWIND nodes AS n
    RETURN n.node AS node, (n.score / max) AS score
}
// dedup & combine
WITH node, max(score) AS score
ORDER BY score DESC
LIMIT $k
RETURN node, score
"""

def hybrid_search_neo4j(question: str, k: int = 4):
    """
    Hybrid retrieval:
      - Vector search on embedding index 'pdf'
      - Fulltext search on 'ftPdfChunk'
      - Scores normalized and merged
    Also computes cosine similarity for inspection.
    """
    step = time.perf_counter()
    log.info(f"STEP: Hybrid Search — question={question!r}, k={k}")

    question_embedding = embed([question])[0]

    records, summary, keys = neo4j_driver.execute_query(
        HYBRID_QUERY,
        question_embedding=question_embedding,
        question=question,
        k=k,
    )

    hybrid_records = []
    for i, rec in enumerate(records):
        node = rec["node"]
        text = node["text"]
        hybrid_score = rec["score"]          # normalized hybrid score
        node_emb = node.get("embedding")     # assumes embedding stored on node

        # local cosine similarity check
        cos_sim = None
        if node_emb is not None:
            cos_sim = cosine_similarity(question_embedding, node_emb)

        # truncate text for speed
        truncated_text = text[:MAX_DOC_CHARS]

        hybrid_records.append(
            {
                "text": truncated_text,
                "full_text_len": len(text),
                "score": hybrid_score,
                "cosine_similarity": cos_sim,
                "node": node,
            }
        )
        log.info(
            f"HybridHit[{i}]: hybrid_score={hybrid_score:.4f}, "
            f"cos_sim={cos_sim:.4f} full_len={len(text)} trunc_len={len(truncated_text)}"
            if cos_sim is not None
            else f"HybridHit[{i}]: hybrid_score={hybrid_score:.4f}, full_len={len(text)} trunc_len={len(truncated_text)}"
        )

    record_step("Hybrid Search (Neo4j)", step)
    return hybrid_records


# =====================================================
# STEP 4: RAG Prompt Building
# =====================================================
def build_rag_prompt(similar_records, question: str):
    step = time.perf_counter()
    log.info("STEP: Build Prompt — combining docs + question")

    docs = [r["text"] for r in similar_records]
    for i, d in enumerate(docs[:3]):
        log.info(f"Doc[{i}] length (truncated): {len(d)} chars")

    docs_block = "\n\n---\n\n".join(docs)

    system_message = (
        "You are a helpful assistant. You must ONLY use the provided documents. "
        "If the answer is not in them, say you don't know."
    )

    user_message = f"""
Use the following documents to answer the question that will follow:

{docs_block}

---

The question to answer using ONLY the above documents is:
{question}
""".strip()

    log.info(f"User message length: {len(user_message)} chars")
    record_step("Build Prompt", step)
    return system_message, user_message


# =====================================================
# STEP 5: Tokenization + Context Check
# =====================================================
def tokenize_and_check(prompt: str):
    step = time.perf_counter()
    log.info("STEP: Tokenization + Context Check")

    inputs = tokenizer(prompt, return_tensors="pt")
    seq_len = inputs["input_ids"].shape[1]
    pct = seq_len / MAX_CONTEXT_TOKENS * 100

    log.info(f"Prompt tokens: {seq_len}/{MAX_CONTEXT_TOKENS} ({pct:.1f}%)")
    if seq_len > MAX_CONTEXT_TOKENS:
        log.warning("⚠️ Prompt exceeds context window — will be truncated by the model.")
    elif pct > 80:
        log.warning("⚠️ Prompt uses >80% of context — little room for generation.")

    record_step("Tokenization", step)
    return inputs.to(device), seq_len


# =====================================================
# STEP 6: Streaming LLM Inference
# =====================================================
def local_stream(system_message, user_message, max_new_tokens=None):
    step = time.perf_counter()
    max_new_tokens = max_new_tokens or MAX_NEW_TOKENS

    log.info(f"STEP: Inference — starting generation (max_new_tokens={max_new_tokens})")

    prompt = f"<|system|>\n{system_message}\n<|user|>\n{user_message}\n<|assistant|>\n"
    inputs, _ = tokenize_and_check(prompt)

    streamer = TextIteratorStreamer(tokenizer, skip_special_tokens=True)
    gen_kwargs = dict(
        **inputs,
        streamer=streamer,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=0.1,
        pad_token_id=tokenizer.eos_token_id,
    )

    def _gen():
        try:
            model.generate(**gen_kwargs)
        except Exception as e:
            log.exception(f"Generation error: {e}")

    Thread(target=_gen, daemon=True).start()

    t0 = time.perf_counter()
    token_count = 0
    char_count = 0

    for piece in streamer:
        print(piece, end="", flush=True)
        token_count += 1
        char_count += len(piece)

    t1 = time.perf_counter()
    elapsed = t1 - t0
    tps = token_count / elapsed if elapsed else 0.0

    log.info(f"\nGenerated tokens: {token_count}, chars: {char_count}")
    log.info(f"Inference time: {elapsed:.2f}s → {tps:.2f} tokens/s")

    record_step("Inference", step)


# =====================================================
# STEP 7: Full Hybrid RAG Wrapper
# =====================================================
def answer_with_hybrid_rag(question: str, k: int = 4):
    """
    End-to-end:
      1) embed question
      2) hybrid retrieval (vector + fulltext) from Neo4j
      3) build RAG prompt
      4) stream answer from HF LLM (TinyLlama or Phi-3)
    """
    log.info("STEP: Pipeline Start — Hybrid RAG Query")
    similar_hybrid_records = hybrid_search_neo4j(question, k=k)

    if not similar_hybrid_records:
        log.warning("No hybrid records returned from Neo4j.")
        return

    system_msg, user_msg = build_rag_prompt(similar_hybrid_records, question)

    print("\n📌 QUESTION:", question)
    print("\n🧠 ANSWER (Hybrid RAG):\n")
    local_stream(system_msg, user_msg)


# =====================================================
# Example usage
# =====================================================
if __name__ == "__main__":
    q = "Explain the main idea of the document."
    answer_with_hybrid_rag(q, k=3)   # small k for speed
    print_summary()


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

ResultConsumedError: The result has been consumed. Fetch all needed records before calling Result.consume().

In [ ]:
# # --- Push changes back to GitHub (optional) ---
# GITHUB_TOKEN = "your_token_here"  # <-- Add a Personal Access Token (classic) with repo access

# !git config --global user.email "your_email@example.com"
# !git config --global user.name "Sathish Kumar"

# !git add .
# !git commit -m "Colab updates" || echo "No changes to commit"
# !git push https://$GITHUB_TOKEN@github.com/SathishKumarAI/kg-rag.git main
